# Interactive Decision Tree - Oracle ve Sample Data Demo

Bu notebook iki akisi gosterir:

1. Notebook icinde uretilen sample pandas `DataFrame` ile UI'i acmak.
2. Oracle'a farkli bir demo dataset yazmak, Oracle'dan geri okumak ve okunan datayi UI'da kullanmak.

Credential bilgileri `oracle_config/ora_config.ini` icinden okunur; kullanici/sifre ekrana yazdirilmaz ve `oracle_config/` git'e dahil edilmez.


## 0. Kurulum notu

Bu notebook'u repo kokunden calistiriyorsan once bir kez sunu calistir:

```powershell
.\.venv\Scripts\python.exe -m pip install -e ".[notebook,oracle]"
```

Notebook kernel'in bu `.venv` degilse, asagidaki `%pip install -e ".[notebook,oracle]"` satirini acip bir kez calistirabilirsin.


In [ ]:
# Import hatasi alirsan once kernel'in `interactive_decision_tree_env (.venv)` oldugunu kontrol et.
# Gerekirse bu satiri acip bir kez calistir:
# %pip install -e ".[notebook,oracle]"

import sys
from configparser import ConfigParser
from pathlib import Path
from urllib.parse import quote_plus

IGNORED_ROOT_PARTS = {".trash", "trash", ".trash-1000", ".trash-1001", "$recycle.bin"}


def is_ignored_project_candidate(path: Path) -> bool:
    return any(part.lower() in IGNORED_ROOT_PARTS for part in path.parts)


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "interactive_decision_tree").is_dir()


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for path in (cwd, *cwd.parents):
        candidates.append(path)
        candidates.append(path / "interactive_decision_tree")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if is_ignored_project_candidate(candidate):
            continue
        if looks_like_project_root(candidate):
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("python_executable:", sys.executable)
print("project_root:", PROJECT_ROOT)

try:
    import numpy as np
    import pandas as pd
    from sqlalchemy import create_engine, text
    from interactive_decision_tree import launch_tree, launch_tree_sql
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Notebook kernel'inde eksik paket var veya yanlis kernel secili. "
        "VS Code/Jupyter kernel olarak `interactive_decision_tree_env (.venv)` sec. "
        f"Aktif Python: {sys.executable}. Eksik modul: {exc.name}. "
        'Gerekirse bu hucreden once `%pip install -e \".[notebook,oracle]\"` calistir.'
    ) from exc


## 0.1 UI link ayarlari

Lokal makinede default ayarlar yeterli. OpenShift/Jupyter proxy veya route kullaniyorsan `APP_BASE_URL`, `APP_HOST` ve `APP_SCHEME` alanlarini bu hucrede degistir.


In [ ]:
import inspect
from urllib.parse import urlsplit, urlunsplit

APP_PORT = 8501
APP_START_SERVER = True
APP_OPEN_BROWSER = True

# OpenShift/Jupyter proxy icin ornek:
# APP_START_SERVER = False
# APP_OPEN_BROWSER = False
# APP_BASE_URL = "https://<notebook-host>/notebook/<workspace>/proxy/8501/"
APP_BASE_URL = ""

# Route veya farkli host icin ornek:
# APP_HOST = "interactive-tree.apps.internal"
# APP_SCHEME = "https"
APP_HOST = "localhost"
APP_SCHEME = "http"


def callable_accepts_kwargs(func) -> bool:
    signature = inspect.signature(func)
    return any(param.kind == inspect.Parameter.VAR_KEYWORD for param in signature.parameters.values())


def filter_supported_kwargs(func, kwargs: dict) -> dict:
    if callable_accepts_kwargs(func):
        return kwargs
    supported = inspect.signature(func).parameters
    return {key: value for key, value in kwargs.items() if key in supported}


def ui_launch_kwargs(func=launch_tree) -> dict:
    # Sadece server davranisi argumanlarini geciyoruz. URL host/proxy donusumu
    # format_ui_url() ile yapiliyor; boylece eski launch_tree surumlerinde de
    # `unexpected keyword argument host` hatasi alinmiyor.
    return filter_supported_kwargs(
        func,
        {
            "port": APP_PORT,
            "start_server": APP_START_SERVER,
            "open_browser": APP_OPEN_BROWSER,
        },
    )


def append_query(base_url: str, query: str) -> str:
    parts = urlsplit(base_url)
    path = parts.path or "/"
    merged_query = f"{parts.query}&{query}" if parts.query else query
    return urlunsplit((parts.scheme, parts.netloc, path, merged_query, parts.fragment))


def format_ui_url(url: str) -> str:
    parsed = urlsplit(url)
    query = parsed.query
    if APP_BASE_URL:
        return append_query(APP_BASE_URL, query)
    if APP_HOST != "localhost" or APP_SCHEME != "http":
        host = APP_HOST if "://" not in APP_HOST else urlsplit(APP_HOST).netloc
        netloc = host if ":" in host else f"{host}:{APP_PORT}"
        base_url = f"{APP_SCHEME}://{netloc}/"
        return append_query(base_url, query)
    return url


ui_launch_kwargs(), format_ui_url(f"http://localhost:{APP_PORT}/?data_id=demo&work_id=demo")


## 1. Notebook icinde sample DataFrame uretme

Bu bolum dis kaynaga baglanmadan RAM'de sample data uretir ve UI'a aktarir.
Sample 2.000.000 satirdir ve Data Setup / WOE testleri icin numeric + categorical missing, special value, yuksek/dusuk univariate IV-Gini karisimi ve 30+ feature icerir.


In [ ]:
rng = np.random.default_rng(20260518)
n = 2_000_000
customer_id = np.arange(10_000_001, 10_000_001 + n, dtype=np.int64)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


latent_risk = rng.normal(0, 1, size=n)
macro_noise = rng.normal(0, 0.55, size=n)

age = rng.integers(21, 74, size=n).astype(float)
income = (62_000 - 11_000 * latent_risk + rng.normal(0, 13_000, size=n)).clip(8_000, 180_000).round(2)
tenure_months = (70 - 18 * latent_risk + rng.normal(0, 26, size=n)).clip(0, 180).round().astype(float)
utilization_rate = sigmoid(-0.25 + 0.95 * latent_risk + rng.normal(0, 0.55, size=n)).round(4)
delinquency_3m = rng.poisson(np.exp(-0.35 + 0.72 * latent_risk)).clip(0, 8).astype(float)
max_dpd_12m = (8 + 19 * latent_risk + rng.normal(0, 13, size=n)).clip(0, 120).round().astype(float)
avg_dpd_6m = (2 + 8 * latent_risk + rng.normal(0, 6, size=n)).clip(0, 70).round(1)
credit_card_limit = (32_000 - 4_500 * latent_risk + rng.normal(0, 10_000, size=n)).clip(1_000, 90_000).round(2)
loan_to_income = sigmoid(-0.6 + 0.75 * latent_risk + rng.normal(0, 0.5, size=n)).round(4)
cash_flow_ratio = (1.45 - 0.32 * latent_risk + rng.normal(0, 0.28, size=n)).clip(0.05, 3.5).round(4)
recent_inquiry_count = rng.poisson(np.exp(-0.2 + 0.55 * latent_risk)).clip(0, 12).astype(float)
past_due_amount = (np.exp(7.2 + 0.75 * latent_risk + rng.normal(0, 0.85, size=n))).clip(0, 120_000).round(2)
savings_balance = (np.exp(9.4 - 0.55 * latent_risk + rng.normal(0, 0.9, size=n))).clip(0, 250_000).round(2)
external_score = (650 - 72 * latent_risk + rng.normal(0, 45, size=n)).clip(300, 850).round().astype(float)
debt_service_ratio = sigmoid(-0.45 + 0.7 * latent_risk + rng.normal(0, 0.45, size=n)).round(4)
transaction_volatility = (0.28 + 0.2 * latent_risk + rng.normal(0, 0.14, size=n)).clip(0.01, 1.5).round(4)
salary_variation = (0.12 + 0.18 * latent_risk + rng.normal(0, 0.12, size=n)).clip(0, 1.2).round(4)
mobile_login_count = rng.poisson(np.exp(2.5 - 0.25 * latent_risk)).clip(0, 80).astype(float)
special_limit_code = (credit_card_limit / 1_000 + rng.normal(0, 6, size=n)).clip(0, 120).round(2)
special_dpd_code = (max_dpd_12m + rng.normal(0, 8, size=n)).clip(0, 150).round(1)

segment = np.select(
    [latent_risk > 0.9, latent_risk > 0.25, latent_risk < -0.8],
    ["D", "C", "A"],
    default="B",
).astype(object)
channel = rng.choice(["branch", "mobile", "web", "call_center"], size=n, p=[0.26, 0.33, 0.29, 0.12]).astype(object)
region = rng.choice(["marmara", "ege", "akdeniz", "ic_anadolu", "karadeniz"], size=n).astype(object)
employment_type = np.select(
    [latent_risk > 0.75, latent_risk < -0.65],
    ["temporary", "payroll"],
    default=rng.choice(["payroll", "self_employed", "retired"], size=n),
).astype(object)
product_type = rng.choice(["cash_loan", "credit_card", "overdraft", "mortgage", "auto"], size=n).astype(object)
risk_band_hint = pd.Series(
    pd.cut(
        latent_risk + rng.normal(0, 0.35, size=n),
        bins=[-np.inf, -0.75, 0.15, 0.85, np.inf],
        labels=["green", "yellow", "orange", "red"],
    )
).astype(object).to_numpy()
collection_status = np.select(
    [latent_risk > 1.15, latent_risk > 0.55, latent_risk < -0.75],
    ["legal_watch", "soft_call", "clean"],
    default="monitor",
).astype(object)
collateral_type = np.select(
    [latent_risk < -0.7, latent_risk > 0.9],
    ["secured", "none"],
    default=rng.choice(["none", "vehicle", "cash", "guarantor"], size=n),
).astype(object)
campaign_group = rng.choice(["control", "campaign_a", "campaign_b", "campaign_c"], size=n).astype(object)
device_type = rng.choice(["ios", "android", "desktop", "unknown_device"], size=n).astype(object)

sample_df = pd.DataFrame(
    {
        "customer_id": customer_id,
        "age": age,
        "income": income,
        "tenure_months": tenure_months,
        "utilization_rate": utilization_rate,
        "delinquency_3m": delinquency_3m,
        "max_dpd_12m": max_dpd_12m,
        "avg_dpd_6m": avg_dpd_6m,
        "credit_card_limit": credit_card_limit,
        "loan_to_income": loan_to_income,
        "cash_flow_ratio": cash_flow_ratio,
        "recent_inquiry_count": recent_inquiry_count,
        "past_due_amount": past_due_amount,
        "savings_balance": savings_balance,
        "external_score": external_score,
        "debt_service_ratio": debt_service_ratio,
        "transaction_volatility": transaction_volatility,
        "salary_variation": salary_variation,
        "mobile_login_count": mobile_login_count,
        "special_limit_code": special_limit_code,
        "special_dpd_code": special_dpd_code,
        "noise_num_01": rng.normal(0, 1, size=n).round(4),
        "noise_num_02": rng.uniform(0, 100, size=n).round(3),
        "noise_num_03": rng.poisson(3, size=n).astype(float),
        "noise_num_04": rng.beta(2, 5, size=n).round(4),
        "segment": segment,
        "channel": channel,
        "region": region,
        "employment_type": employment_type,
        "product_type": product_type,
        "risk_band_hint": risk_band_hint,
        "collection_status": collection_status,
        "collateral_type": collateral_type,
        "campaign_group": campaign_group,
        "device_type": device_type,
        "special_cat_code": rng.choice(["std", "alt", "legacy"], size=n, p=[0.55, 0.3, 0.15]).astype(object),
        "noise_cat_01": rng.choice(["N1", "N2", "N3", "N4"], size=n).astype(object),
        "noise_cat_02": rng.choice(["blue", "green", "red"], size=n).astype(object),
        "noise_cat_03": rng.choice(["x", "y"], size=n).astype(object),
        "noise_cat_04": rng.choice(["north", "south", "east", "west"], size=n).astype(object),
    }
)

sample_logit = (
    -0.25
    + 1.05 * latent_risk
    + 1.05 * (sample_df["delinquency_3m"] >= 2).astype(float)
    + 0.85 * (sample_df["max_dpd_12m"] >= 30).astype(float)
    + 0.75 * (sample_df["utilization_rate"] >= 0.72).astype(float)
    + 0.65 * sample_df["segment"].isin(["C", "D"]).astype(float)
    + 0.55 * sample_df["collection_status"].isin(["soft_call", "legal_watch"]).astype(float)
    - 0.55 * (sample_df["external_score"] >= 700).astype(float)
    + macro_noise
)
sample_df["risk_flag"] = np.where(rng.random(n) < sigmoid(sample_logit), "high_risk", "low_risk")

numeric_special_values = {
    "income": -999.0,
    "external_score": -1.0,
    "special_limit_code": 999999.0,
    "special_dpd_code": -999.0,
    "noise_num_02": -999.0,
}
categorical_special_values = {
    "segment": "UNKNOWN",
    "channel": "NO_INFO",
    "employment_type": "SPECIAL_CASE",
    "special_cat_code": "SPECIAL_CASE",
    "noise_cat_02": "UNKNOWN",
}


def inject_numeric_quality(frame: pd.DataFrame, column: str, special_value: float, missing_rate=0.045, special_rate=0.035):
    missing_mask = rng.random(n) < missing_rate
    special_mask = (rng.random(n) < special_rate) & ~missing_mask
    frame.loc[missing_mask, column] = np.nan
    frame.loc[special_mask, column] = special_value


def inject_categorical_quality(frame: pd.DataFrame, column: str, special_value: str, missing_rate=0.045, special_rate=0.035):
    missing_mask = rng.random(n) < missing_rate
    special_mask = (rng.random(n) < special_rate) & ~missing_mask
    frame.loc[missing_mask, column] = None
    frame.loc[special_mask, column] = special_value


for column, special_value in numeric_special_values.items():
    inject_numeric_quality(sample_df, column, special_value)
for column, special_value in categorical_special_values.items():
    inject_categorical_quality(sample_df, column, special_value)

sample_features = [column for column in sample_df.columns if column not in {"risk_flag", "customer_id"}]
sample_numeric_features = sample_df[sample_features].select_dtypes(include="number").columns.tolist()
sample_categorical_features = [column for column in sample_features if column not in sample_numeric_features]
for column in sample_categorical_features:
    sample_df[column] = sample_df[column].astype("category")


def auc_from_scores(y_true: pd.Series, score: pd.Series) -> float:
    y = y_true.astype(int).to_numpy()
    scores = pd.Series(score).astype(float)
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = scores.rank(method="average").to_numpy()
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def metric_bins(series: pd.Series, max_bins: int = 5) -> pd.Series:
    labels = pd.Series(index=series.index, dtype="object")
    labels[series.isna()] = "__MISSING__"
    if pd.api.types.is_numeric_dtype(series):
        special_values = {-999.0, 999999.0, -1.0}
        special_mask = series.isin(special_values)
        labels[special_mask] = "__SPECIAL__=" + series[special_mask].astype(str)
        normal = series.notna() & ~special_mask
        if normal.any():
            try:
                labels[normal] = pd.qcut(series[normal], q=min(max_bins, series[normal].nunique()), duplicates="drop").astype(str)
            except ValueError:
                labels[normal] = series[normal].astype(str)
    else:
        labels = series.astype("object").where(series.notna(), "__MISSING__").astype(str)
    return labels.fillna("__OTHER__")


def univariate_iv_gini(frame: pd.DataFrame, feature: str, target: str = "risk_flag") -> dict:
    y = (frame[target] == "high_risk").astype(int)
    bins = metric_bins(frame[feature])
    stats = pd.DataFrame({"bin": bins, "event": y}).groupby("bin", dropna=False)["event"].agg(["sum", "count"])
    stats["non_event"] = stats["count"] - stats["sum"]
    event_dist = (stats["sum"] + 0.5) / (stats["sum"].sum() + 0.5 * len(stats))
    non_event_dist = (stats["non_event"] + 0.5) / (stats["non_event"].sum() + 0.5 * len(stats))
    woe = np.log(event_dist / non_event_dist)
    iv = float(((event_dist - non_event_dist) * woe).sum())
    event_rate_by_bin = stats["sum"] / stats["count"].replace(0, np.nan)
    gini = abs(2 * auc_from_scores(y, bins.map(event_rate_by_bin)) - 1)
    return {
        "feature": feature,
        "type": "numeric" if pd.api.types.is_numeric_dtype(frame[feature]) else "categorical",
        "iv": iv,
        "gini": float(gini),
        "missing_count": int(frame[feature].isna().sum()),
        "special_count": int(bins.astype(str).str.startswith("__SPECIAL__").sum())
        if pd.api.types.is_numeric_dtype(frame[feature])
        else int(frame[feature].isin(["UNKNOWN", "NO_INFO", "SPECIAL_CASE"]).sum()),
        "unique_count": int(frame[feature].nunique(dropna=True)),
    }


sample_univariate = pd.DataFrame([univariate_iv_gini(sample_df, feature) for feature in sample_features]).sort_values(
    ["gini", "iv"], ascending=False
)

print(
    {
        "rows": len(sample_df),
        "feature_count": len(sample_features),
        "numeric_features": len(sample_numeric_features),
        "categorical_features": len(sample_categorical_features),
        "target_distribution": sample_df["risk_flag"].value_counts().to_dict(),
    }
)
print("Top univariate IV/Gini variables")
display(sample_univariate.head(10))
print("Low univariate IV/Gini variables")
display(sample_univariate.tail(10))

sample_df.head()


In [ ]:
sample_url = format_ui_url(
    launch_tree(
        sample_df,
        target="risk_flag",
        features=sample_features,
        session_name="Notebook synthetic IV/Gini sample data",
        **ui_launch_kwargs(launch_tree),
    )
)
sample_url


## 2. Oracle baglantisini acma

Bu hucre `oracle_config/ora_config.ini` veya `ora_config/ora_config.ini` dosyasini okur, SQLAlchemy Oracle URL'ini bellekte olusturur ve `select 1 from dual` ile baglantiyi test eder.


In [ ]:
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

oracle_config_candidates = [
    PROJECT_ROOT / "oracle_config" / "ora_config.ini",
    PROJECT_ROOT / "ora_config" / "ora_config.ini",
    Path.cwd().resolve() / "oracle_config" / "ora_config.ini",
    Path.cwd().resolve() / "ora_config" / "ora_config.ini",
    Path.cwd().resolve().parent / "oracle_config" / "ora_config.ini",
    Path.cwd().resolve().parent / "ora_config" / "ora_config.ini",
]

oracle_config_path = next((path for path in oracle_config_candidates if path.exists()), None)
if oracle_config_path is None:
    searched_paths = "\n".join(str(path) for path in oracle_config_candidates)
    raise FileNotFoundError(
        "oracle_config/ora_config.ini bulunamadi. Aranan yollar:\n" + searched_paths
    )

oracle_section = "ORA_PRD_ZTUSER"
parser = ConfigParser()
parser.read(oracle_config_path, encoding="utf-8")
if oracle_section not in parser:
    raise KeyError(f"INI icinde bolum bulunamadi: {oracle_section}")

cfg = parser[oracle_section]
oracle_url = (
    "oracle+oracledb://"
    f"{quote_plus(cfg['user'])}:{quote_plus(cfg['password'])}"
    f"@{cfg['host']}:{cfg.get('port', '1521')}/?service_name={quote_plus(cfg['service_name'])}"
)

engine = create_engine(oracle_url)
try:
    with engine.connect() as conn:
        ok_value = conn.execute(text("select 1 as ok from dual")).scalar_one()
finally:
    engine.dispose()

print({
    "section": oracle_section,
    "config_path": str(oracle_config_path),
    "connection": "ok",
    "select_1": int(ok_value),
})



## 3. Oracle icin farkli demo data uretme

Bu dataset lokal sample datadan farklidir. Sonraki hucrede Oracle'a tablo olarak yazilir.


In [ ]:
oracle_rng = np.random.default_rng(42)
oracle_n = 320

oracle_demo_df = pd.DataFrame(
    {
        "CUSTOMER_ID": np.arange(1, oracle_n + 1),
        "AGE": oracle_rng.integers(22, 76, size=oracle_n),
        "INCOME": oracle_rng.normal(64_000, 22_000, size=oracle_n).clip(15_000, 180_000).round(2),
        "TENURE_MONTHS": oracle_rng.integers(0, 144, size=oracle_n),
        "SEGMENT": oracle_rng.choice(["SME", "MASS", "AFFLUENT", "YOUNG"], size=oracle_n),
        "CHANNEL": oracle_rng.choice(["BRANCH", "MOBILE", "WEB", "CALL_CENTER"], size=oracle_n),
        "REGION": oracle_rng.choice(["MARMARA", "EGE", "AKDENIZ", "IC_ANADOLU", "KARADENIZ"], size=oracle_n),
        "UTILIZATION": oracle_rng.beta(2.2, 4.5, size=oracle_n).round(4),
    }
)

oracle_score = (
    (oracle_demo_df["INCOME"] < 48_000).astype(int)
    + (oracle_demo_df["TENURE_MONTHS"] < 24).astype(int)
    + oracle_demo_df["SEGMENT"].isin(["SME", "YOUNG"]).astype(int)
    + (oracle_demo_df["CHANNEL"] == "MOBILE").astype(int)
    + (oracle_demo_df["UTILIZATION"] > 0.52).astype(int)
    - (oracle_demo_df["AGE"] > 60).astype(int)
)
oracle_demo_df["RISK_FLAG"] = np.where(oracle_score >= 3, "high_risk", "low_risk")

oracle_demo_df.head()


## 4. Oracle'a yazma

Bu hucre mevcut kullanicinin schema'sinda `IDT_DEMO_TREE_DATA` tablosunu olusturur veya replace eder. Calistirmadan once bu tablo adinin senin ortaminda uygun oldugunu kontrol et.


In [ ]:
import re

from sqlalchemy import Integer, Numeric, String
from sqlalchemy.exc import SQLAlchemyError

oracle_table_name = "idt_demo_tree_data"
if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,29}", oracle_table_name):
    raise ValueError("Oracle tablo adi harf, rakam ve underscore icermeli; 30 karakteri asmamali.")

oracle_write_df = oracle_demo_df.copy().where(pd.notna(oracle_demo_df), None)
oracle_sql_dtypes = {
    "CUSTOMER_ID": Integer(),
    "AGE": Integer(),
    "INCOME": Numeric(18, 2),
    "TENURE_MONTHS": Integer(),
    "SEGMENT": String(30),
    "CHANNEL": String(30),
    "REGION": String(30),
    "UTILIZATION": Numeric(10, 4),
    "RISK_FLAG": String(30),
}


def oracle_error_text(exc: BaseException) -> str:
    orig = getattr(exc, "orig", "")
    return f"{exc} {orig}".upper()


def oracle_table_exists(conn, table_name: str) -> bool:
    return bool(
        conn.execute(
            text("select count(*) from user_tables where table_name = :table_name"),
            {"table_name": table_name.upper()},
        ).scalar_one()
    )


engine = create_engine(oracle_url)
row_count = None
write_mode = "create"
try:
    with engine.connect() as conn:
        table_exists = oracle_table_exists(conn, oracle_table_name)

    if table_exists:
        try:
            with engine.begin() as conn:
                conn.execute(text(f"drop table {oracle_table_name} purge"))
            table_exists = False
            write_mode = "drop_create"
        except SQLAlchemyError as exc:
            error_text = oracle_error_text(exc)
            if "ORA-00942" in error_text:
                table_exists = False
                write_mode = "create"
            elif "ORA-01031" in error_text:
                with engine.begin() as conn:
                    conn.execute(text(f"delete from {oracle_table_name}"))
                write_mode = "delete_append"
            else:
                raise

    oracle_write_df.to_sql(
        oracle_table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        dtype=oracle_sql_dtypes,
    )
    with engine.connect() as conn:
        row_count = conn.execute(text(f"select count(*) from {oracle_table_name}")).scalar_one()
finally:
    engine.dispose()

print({"table": oracle_table_name, "rows_written": int(row_count), "mode": write_mode})


## 5. Oracle query ile okuyup UI'da kullanma

Bu bolum tabloyu dogrudan secmek yerine SQL query yazarak Oracle'dan DataFrame okur ve UI'a aktarir. Oracle/pandas kolonlari lowercase dondurebildigi icin target kolonu case-insensitive cozulur; ornegin `RISK_FLAG` istersen ve DataFrame'de `risk_flag` donerse otomatik onu kullanir.


In [ ]:
oracle_read_query = f"""
select
    CUSTOMER_ID,
    AGE,
    INCOME,
    TENURE_MONTHS,
    SEGMENT,
    CHANNEL,
    REGION,
    UTILIZATION,
    RISK_FLAG
from {oracle_table_name}
"""


def resolve_df_column(df: pd.DataFrame, requested: str) -> str:
    if requested in df.columns:
        return requested
    matches = [column for column in df.columns if str(column).casefold() == requested.casefold()]
    if len(matches) == 1:
        return str(matches[0])
    raise ValueError(f"Kolon bulunamadi: {requested}. Donen kolonlar: {list(df.columns)}")


def launch_oracle_query_to_ui(query: str, target: str, session_name: str = "Oracle query data") -> str:
    engine = create_engine(oracle_url)
    try:
        oracle_ui_df = pd.read_sql_query(text(query), engine)
    finally:
        engine.dispose()

    target_column = resolve_df_column(oracle_ui_df, target)
    print({"oracle_columns": list(oracle_ui_df.columns), "target_used": target_column, "rows": len(oracle_ui_df)})
    return format_ui_url(
        launch_tree(
            oracle_ui_df,
            target=target_column,
            session_name=session_name,
            **ui_launch_kwargs(launch_tree),
        )
    )


oracle_query_url = launch_oracle_query_to_ui(
    oracle_read_query,
    target="RISK_FLAG",
    session_name="Oracle query demo data",
)
oracle_query_url


## 6. Final agaci notebook'a geri yukleme

UI'da agaci finalize ettikten sonra en alttaki `Tree export` bolumunden `Download runnable tree pickle` ile dosyayi indir. Sonra ayni veya baska bir notebook'ta asagidaki gibi acabilirsin.

Not: Pickle dosyalarini sadece kendi urettigin guvenilir dosyalardan yukle.


In [ ]:
from interactive_decision_tree import load_tree_pickle, load_tree_json, score_tree_payload

# UI export pickle dosyasini Downloads klasorune indirdiysen:
tree_pickle_path = Path.home() / "Downloads" / "interactive_entropy_tree_runnable.pkl"
tree_payload = load_tree_pickle(tree_pickle_path)

print("loaded_tree_path:", tree_pickle_path)
print("loaded_tree_modified:", pd.Timestamp.fromtimestamp(tree_pickle_path.stat().st_mtime))
print("loaded_tree_nodes:", tree_payload.get("node_count"))

# JSON indirdiysen:
# tree_payload = load_tree_json(Path.home() / "Downloads" / "interactive_entropy_tree_runnable.json")

# tree_payload.keys()
# tree_payload["tree"]
# tree_payload["metrics"]


## 7. Tek musteri icin skorlama

Bu hucrede musteri degiskenlerini notebook icinde yaratip yukledigimiz gercek pickle/JSON payload ile skorlariz. `tree_payload` bir onceki hucrede UI exportundan yuklenmis olmalidir.

`prediction` global bir threshold'a gore degil, musteri hangi leaf'e dustuyse o leaf'in cogunluk sinifina gore gelir. Binary target icin `positive_class_probability` leaf icindeki positive class oranidir; `prediction_probability` tahmin edilen sinifin leaf icindeki oranidir.


In [ ]:
if "tree_payload" not in globals():
    raise RuntimeError("Once UI export pickle/JSON dosyasini tree_payload degiskenine yukle.")

payload_features = {str(feature) for feature in tree_payload.get("features", [])}

if {"AGE", "INCOME", "TENURE_MONTHS", "SEGMENT", "CHANNEL", "REGION", "UTILIZATION"}.issubset(payload_features):
    one_customer = {
        "CUSTOMER_ID": 999001,
        "AGE": 34,
        "INCOME": 35_000,
        "TENURE_MONTHS": 12,
        "SEGMENT": "SME",
        "CHANNEL": "MOBILE",
        "REGION": "MARMARA",
        "UTILIZATION": 0.62,
    }
else:
    one_customer = {
        "age": 34,
        "income": 35_000,
        "tenure_months": 12,
        "segment": "C",
        "channel": "mobile",
        "region": "marmara",
    }

score_result = score_tree_payload(tree_payload, one_customer)

print("prediction:", score_result["prediction"])
print("prediction_proba:", score_result["prediction_probability"])
print("positive_class:", score_result["positive_class"])
print("positive_class_proba:", score_result["positive_class_probability"])
print("leaf_node_id:", score_result["leaf_node_id"])
print("leaf_path:", score_result["leaf_path"])
if score_result.get("exported_leaf_path") != score_result.get("leaf_path"):
    print("exported_leaf_path:", score_result.get("exported_leaf_path"))
display(pd.DataFrame([score_result["class_probabilities"]]))
pd.DataFrame(score_result["trace"])
